# SpeechBrain

A self-contained refresher on **SpeechBrain** — an open-source, PyTorch-based **all-in-one toolkit** for speech and audio: ASR, speaker recognition, speech enhancement, source separation, diarization, spoken-language understanding, and TTS, all behind one consistent API.

**Domain:** Speech & Audio  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**SpeechBrain** packages a broad span of speech tasks into a single, hackable PyTorch framework. Where most libraries do one thing (Whisper = ASR, pyannote = diarization, Demucs = separation), SpeechBrain gives you **one toolkit and one mental model** across ASR, **speaker verification/identification** (ECAPA-TDNN, x-vectors), **speech enhancement & separation** (SepFormer, MetricGAN+), **VAD**, **diarization**, **SLU**, **language ID**, and **TTS** (Tacotron2 + HiFi-GAN). Models and recipes ship for VoxCeleb, LibriSpeech, WSJ, CommonVoice, and more, with **pretrained checkpoints on Hugging Face** you can run in three lines.

**The problem it solves.** Research and prototyping in speech usually means stitching together five incompatible repos, each with its own data format, training loop, and config style. SpeechBrain standardizes all of it: a single `Brain` training loop, a single **YAML (HyperPyYAML)** way to declare experiments, and a single family of `Pretrained` inference interfaces. The design priority is **readability and ease of modification** — it is the framework you reach for when you want to *understand* and *fine-tune* a speech model, not just call an API.

**When to reach for it.** Prototyping or fine-tuning speaker/ASR/enhancement models; teaching and reproducible research; building a custom pipeline that mixes several speech tasks. **When not to:** if you just need one production transcript on a laptop, `faster-whisper` is lighter; for a turnkey hosted API, a cloud STT/TTS service is less work.

## 2. Mental Model

Think of SpeechBrain as **"PyTorch Lightning specialized for speech, plus a config language that *is* your experiment."** Two abstractions carry almost everything:

```
   HyperPyYAML (.yaml)                         Brain class (Python)
   ──────────────────────                      ─────────────────────────────────
   model:   !new:MyNet {...}                   class ASR(sb.Brain):
   opt:     !name:Adam                             compute_forward(batch, stage) -> preds
   lr:      !ref <base_lr>                         compute_objectives(preds, batch, stage) -> loss
   tokenizer: !new:...                          brain = ASR(modules, opt, hparams)
        │  (objects, not just strings)          brain.fit(epochs, train_set, valid_set)
        ▼                                                   │  on_stage_start / _end hooks
   load_hyperpyyaml(f) -> dict of live objects  ───────────►│  checkpoints, schedulers, logging
```

The **YAML builds your objects** (`!new:` instantiates a class, `!ref` wires values together), and the **`Brain` class owns the training loop** — you only override `compute_forward` (data → predictions) and `compute_objectives` (predictions → loss). Everything else (batching, AMP, checkpointing, multi-GPU, schedulers) is handled. For inference you skip all of that and use a **`Pretrained` interface** (`EncoderClassifier`, `EncoderDecoderASR`, `SpeakerRecognition`, `SepformerSeparation`) that loads a HF checkpoint and exposes a task verb like `.transcribe_file()` or `.verify_batch()`.

## 3. Key Concepts

- **`Brain` class.** The training-loop engine. You subclass `speechbrain.Brain` and implement just `compute_forward` (batch → predictions) and `compute_objectives` (predictions → loss). `brain.fit(...)` runs the epochs; `on_stage_start/end` hooks handle metrics and checkpointing. This is SpeechBrain's analogue to a Lightning `LightningModule`.
- **HyperPyYAML.** A superset of YAML where `!new:pkg.Class {args}` *instantiates* an object, `!name:pkg.func` references a callable, and `!ref <key>` substitutes another value. `load_hyperpyyaml(file, overrides)` returns a dict of **live Python objects** — your whole experiment is declarative and overridable from the command line.
- **`Pretrained` interfaces.** Inference wrappers around a HF checkpoint. `EncoderClassifier` (language ID / speaker ID), `SpeakerRecognition` (verification), `EncoderDecoderASR` / `EncoderASR` (transcription), `SepformerSeparation`, `SpectralMaskEnhancement`. `from_hparams(source=..., savedir=...)` downloads and caches the model.
- **Speaker embedding + cosine scoring.** Speaker verification = run both utterances through an encoder (**ECAPA-TDNN**) to get fixed-length **embeddings**, then **cosine-similarity** them and threshold. Same/different-speaker is a single number vs a decision boundary (see Example 1).
- **`DynamicItemDataset` & pipelines.** Data is a dataset of items plus **dynamic pipelines** — functions decorated to take input keys and `@provides` output keys (e.g. read a wav path → waveform → features) — evaluated lazily per batch.
- **Recipes.** Each task/dataset pair ships a `recipes/<Dataset>/<task>/` folder with `train.py` + `hparams.yaml` you can run or adapt — the canonical way to reproduce or fine-tune.

## 4. Setup

SpeechBrain is a pip install on top of PyTorch (CPU works; GPU helps for training):

```bash
pip install speechbrain        # pulls torch, torchaudio, huggingface_hub, hyperpyyaml
```

Pretrained inference is then a few lines — e.g. speaker verification:

```python
from speechbrain.inference.speaker import SpeakerRecognition
verifier = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained/ecapa",
)
score, prediction = verifier.verify_files("a.wav", "b.wav")  # score in [-1,1], bool same-speaker
```

The first two worked examples below reproduce SpeechBrain's **core mechanics with plain NumPy** so they run anywhere on CPU in milliseconds (no model download). The third shows the **real** SpeechBrain API, gated behind an env var + import check so the notebook still executes end-to-end without the heavy dependency.

In [ ]:
import json
import os

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

## 5. Worked Examples

### Example 1 — Speaker verification by cosine scoring (the heart of `SpeakerRecognition`)

A SpeechBrain speaker-verification model (`spkrec-ecapa-voxceleb`) maps each utterance to a fixed-length **embedding**; verification is then just **cosine similarity vs a threshold**. Below we stand in for ECAPA with random-but-consistent embeddings: each speaker has a "voiceprint" centroid, and utterances are that centroid plus noise. Cosine-score within-speaker vs across-speaker pairs and pick a threshold — exactly what `verify_batch` returns.

In [ ]:
def l2norm(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)


def cosine(a, b):
    a, b = l2norm(a), l2norm(b)
    return float(np.sum(a * b, axis=-1))


# Two speakers, each a centroid "voiceprint" in a 192-d embedding space (ECAPA dim).
dim = 192
spk_A = rng.standard_normal(dim)
spk_B = rng.standard_normal(dim)


def utterance(centroid, noise=0.6):
    """An utterance embedding = speaker centroid + per-recording noise."""
    return centroid + noise * rng.standard_normal(dim)


# Genuine pair (same speaker) vs impostor pair (different speakers).
genuine = cosine(utterance(spk_A), utterance(spk_A))
impostor = cosine(utterance(spk_A), utterance(spk_B))
print(f"genuine  (A vs A) cosine = {genuine:+.3f}")
print(f"impostor (A vs B) cosine = {impostor:+.3f}")

# Decision rule: same speaker if score > threshold (tuned on a dev set).
THRESHOLD = 0.25
for name, s in [("genuine", genuine), ("impostor", impostor)]:
    print(f"{name:9s} -> same_speaker={s > THRESHOLD}")

Genuine pairs land well above the threshold, impostors below it. A real ECAPA model just produces far more separable embeddings; the **decision logic is identical** to what we wrote. Sweep many pairs and you get the Equal Error Rate (EER) used to report speaker-verification accuracy:

In [ ]:
N = 2000
gen_scores = np.array([cosine(utterance(spk_A), utterance(spk_A)) for _ in range(N)])
imp_scores = np.array([cosine(utterance(spk_A), utterance(spk_B)) for _ in range(N)])

# EER: the threshold where false-accept rate == false-reject rate.
ts = np.linspace(-1, 1, 400)
far = np.array([(imp_scores > t).mean() for t in ts])   # impostors wrongly accepted
frr = np.array([(gen_scores <= t).mean() for t in ts])  # genuine wrongly rejected
i = np.argmin(np.abs(far - frr))
print(f"EER ~= {(far[i] + frr[i]) / 2:.1%} at threshold {ts[i]:+.3f}")

### Example 2 — The `Brain` training loop, distilled

SpeechBrain's central abstraction is the **`Brain` class**: you implement `compute_forward` and `compute_objectives`, and `fit()` drives epochs, batching, and optimization for you. Here is a minimal pure-NumPy `Brain`-like class that fits a tiny linear regressor, so the **shape of the API** is concrete — the real `speechbrain.Brain` adds AMP, checkpointing, multi-GPU, and schedulers around this same skeleton.

In [ ]:
class MiniBrain:
    """A teaching-sized stand-in for speechbrain.Brain (same method contract)."""

    def __init__(self, modules, lr=0.1):
        self.w = modules["w"]          # the "model" parameters
        self.lr = lr

    def compute_forward(self, batch):
        x, _ = batch
        return x @ self.w              # predictions

    def compute_objectives(self, preds, batch):
        _, y = batch
        return np.mean((preds - y) ** 2)   # MSE loss

    def fit_batch(self, batch):
        x, y = batch
        preds = self.compute_forward(batch)
        loss = self.compute_objectives(preds, batch)
        grad = 2 * x.T @ (preds - y) / len(y)   # manual gradient step
        self.w -= self.lr * grad
        return loss

    def fit(self, epochs, train_batches):
        for ep in range(epochs):
            losses = [self.fit_batch(b) for b in train_batches]
            if ep % 50 == 0 or ep == epochs - 1:
                print(f"epoch {ep:3d}  loss={np.mean(losses):.4f}")


# Toy data: y = 3*x0 - 2*x1 (+ noise). Two minibatches.
true_w = np.array([3.0, -2.0])
X = rng.standard_normal((128, 2))
y = X @ true_w + 0.05 * rng.standard_normal(128)
batches = [(X[:64], y[:64]), (X[64:], y[64:])]

brain = MiniBrain(modules={"w": np.zeros(2)}, lr=0.1)
brain.fit(epochs=200, train_batches=batches)
print("learned w:", np.round(brain.w, 3), " (true:", true_w, ")")

The takeaway is the **contract**, not the math: in real SpeechBrain you subclass `sb.Brain`, return predictions from `compute_forward(batch, stage)` and a loss from `compute_objectives(preds, batch, stage)`, then call `brain.fit(epoch_counter, train_set, valid_set)`. The framework owns everything else.

### Example 3 — Real SpeechBrain speaker verification (gated)

This is the actual SpeechBrain workflow. It's gated behind `RUN_SPEECHBRAIN=1` **and** an import check because `speechbrain` pulls in torch/torchaudio and downloads a checkpoint (~80 MB) — so the notebook still runs end-to-end without it. Set the env var in an environment where SpeechBrain is installed to load `spkrec-ecapa-voxceleb` and verify two WAV files.

In [ ]:
if os.getenv("RUN_SPEECHBRAIN") == "1":
    try:
        import torchaudio  # noqa: F401  (SpeechBrain audio I/O backend)
        from speechbrain.inference.speaker import SpeakerRecognition

        verifier = SpeakerRecognition.from_hparams(
            source="speechbrain/spkrec-ecapa-voxceleb",
            savedir="pretrained/ecapa",
        )
        # verify_files returns (score in [-1,1], bool prediction).
        score, same = verifier.verify_files("utt_a.wav", "utt_b.wav")
        print(f"cosine score = {float(score):+.3f}  same_speaker = {bool(same)}")
    except Exception as exc:  # pragma: no cover - optional heavy dependency
        print("SpeechBrain path not runnable here:", type(exc).__name__, exc)
else:
    print("Set RUN_SPEECHBRAIN=1 (with speechbrain installed) to run real inference.")
    print("API shape: SpeakerRecognition.from_hparams(source=...).verify_files(a, b)")

## 6. Gotchas & Pitfalls

- **Import paths moved.** Modern SpeechBrain (≥ 1.0) uses `speechbrain.inference.*` (e.g. `speechbrain.inference.speaker.SpeakerRecognition`). Older tutorials use `speechbrain.pretrained.*`, which is deprecated — match the import to your installed version.
- **16 kHz mono, always.** Pretrained speech models expect 16 kHz mono audio. Feed 44.1 kHz or stereo and you get wrong results or shape errors — resample with `torchaudio` first. The inference interfaces resample for you on file paths, but not on raw tensors you pass in.
- **HyperPyYAML is code, not data.** `!new:` runs constructors and `!ref` resolves references *at load time*, in order. A forward reference or a typo'd class path fails when you call `load_hyperpyyaml`, not at runtime — and YAML indentation bugs become silent object-graph bugs. Keep configs small and test them.
- **Checkpoint caching & `savedir`.** `from_hparams(savedir=...)` caches the download; reusing a stale `savedir` across different `source`s, or a read-only dir, causes confusing load errors. Give each model its own `savedir`.
- **It's a framework, not a one-call API.** Training means writing a `train.py` + `hparams.yaml` (start from a recipe). Expecting a single `model.train(data)` call will frustrate you — the power is in the overridable loop, which is also the learning curve.
- **`torchaudio` backend.** Audio loading depends on a working `torchaudio` backend (soundfile/sox). Missing system codecs surface as opaque load failures — install `soundfile` if file reads fail.

## 7. When to Use vs Alternatives

| Option | Trade-off vs SpeechBrain |
|---|---|
| **faster-whisper / WhisperX** | Far lighter for **just transcription**: one pip install, great CPU/GPU ASR, no training loop. Pick it to grab transcripts. SpeechBrain wins when you also need speaker ID, enhancement, separation, or to fine-tune. |
| **NVIDIA NeMo** | Heavier, GPU-first, top-of-leaderboard English ASR (Parakeet/Canary) and a production/Riva deployment path. SpeechBrain is friendlier, more hackable, and broader across non-ASR speech tasks — the better *research/teaching* framework. |
| **pyannote.audio** | Specialized and best-in-class for **diarization/VAD/segmentation**. SpeechBrain does diarization too but pyannote's pipelines are stronger; they're often combined (ECAPA embeddings + pyannote clustering). |
| **HF Transformers (Wav2Vec2, Whisper)** | Huge ecosystem and a uniform `pipeline()` API, but thinner on speaker/enhancement/separation and on a unified *training* story for them. SpeechBrain's `Brain`+YAML recipes make custom speech training more reproducible. |
| **ESPnet** | Comparable scope and strong ASR/TTS recipes, but a steeper, more research-style learning curve. SpeechBrain prioritizes readability and modification. |

## 8. Resources

- **Official docs** — https://speechbrain.readthedocs.io/
- **GitHub (speechbrain/speechbrain)** — https://github.com/speechbrain/speechbrain
- **Tutorials (Colab notebooks)** — https://speechbrain.github.io/tutorial_basics.html
- **Pretrained models on Hugging Face** — https://huggingface.co/speechbrain (e.g. `spkrec-ecapa-voxceleb`, `sepformer-wsj02mix`, `asr-crdnn-rnnlm-librispeech`)
- **HyperPyYAML reference** — https://github.com/speechbrain/HyperPyYAML
- **Paper — "SpeechBrain: A General-Purpose Speech Toolkit" (Ravanelli et al., 2021)** — https://arxiv.org/abs/2106.04624

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def cosine(a, b):
    ...


def equal_error_rate(genuine, impostor):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE